# G08 — Experimental v3 LDM with v-prediction and Min-SNR weighting

Generator **G08 (`08_ldm_v3_sdvae_fromscratch`)** is an alternative from-scratch denoiser trained in an SD-VAE latent space. Relative to G07, it changes the Keras U-Net blocks to a Stable-Diffusion-inspired v3 design, replaces transposed convolutions with nearest-neighbor upsampling followed by 3 × 3 convolutions, uses GroupNorm–SiLU residual blocks with FiLM-style time conditioning, and trains with **v-prediction** plus **Min-SNR-$\gamma$ weighting**. The default codec is the frozen SD 2.1 VAE; an explicit option can instead use the VAE fine-tuned by notebook 03.

For forward-process noise $\epsilon \sim \mathcal{N}(0,I)$,

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,
$$

the v-target is

$$
v_t = \sqrt{\bar{\alpha}_t}\,\epsilon - \sqrt{1-\bar{\alpha}_t}\,x_0.
$$

During sampling, the predicted velocity is converted consistently through

$$
\hat{\epsilon}_\theta(x_t,t)
= \sqrt{\bar{\alpha}_t}\,\hat{v}_\theta(x_t,t)
+ \sqrt{1-\bar{\alpha}_t}\,x_t.
$$

With $\operatorname{SNR}(t)=\bar{\alpha}_t/(1-\bar{\alpha}_t)$ and $\gamma=5$, the implemented v-prediction loss is

$$
\mathcal{L}_v = \mathbb{E}_{t,x_0,\epsilon}\!\left[
\frac{\min(\operatorname{SNR}(t),\gamma)}{\operatorname{SNR}(t)+1}
\left\lVert v_t-\hat{v}_\theta(x_t,t)\right\rVert_2^2
\right].
$$

The same `UNET_VERSION`, `PARAMETERIZATION`, and VAE-source flags are passed to training, checkpoint evaluation, sampling sweeps, and both class-generation paths. This consistency is essential: interpreting a v-prediction checkpoint as an epsilon-prediction checkpoint would invalidate the reverse process and every downstream metric.

G08 is an experimental comparison, not the selected from-scratch generator; G07 holds that role in the downstream protocol. Architectural motivation does not imply empirical superiority. Checkpoint and sampler decisions use finite validation samples, generic image features, and repeated screening without formal uncertainty intervals. Report observed differences with their sample sizes and selection procedure, and avoid causal claims.

Artifacts are isolated under `experiments/diffusers/08_ldm_v3_sdvae_fromscratch/`, `results/2_diffusers/08_ldm_v3_sdvae_fromscratch/`, and `data/synthetic/08_ldm_v3_sdvae_fromscratch/{positive,negative}/`. Latent, training, runtime, generation, and filter manifests are required to preserve the full parameterization lineage.


## 1. Runtime bootstrap and accelerator policy

This cell must run before TensorFlow imports. It establishes deterministic CUDA device ordering, enables TensorFlow memory growth, disables automatic XLA JIT, searches portable `libdevice` locations, and defines separate environments for training and parallel generation. Missing `libdevice` produces an explicit warning so the runtime can be corrected before a long job.

Device visibility should be inherited from the launcher or resolved at runtime; it is not a model hyperparameter and no GPU UUID belongs in the notebook. The environment diagnostics support reproducibility audits but do not alter G08's scientific identity.


In [1]:
# CUDA and XLA setup: run this cell before any TensorFlow import.
# This does not select a GPU; define CUDA_VISIBLE_DEVICES externally when needed.
import os
import sys
from pathlib import Path as _Path

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

_libdevice_candidates = [
    _Path(sys.prefix) / "nvvm" / "libdevice" / "libdevice.10.bc",
    _Path(os.environ["MAMMODIFFUSION_CUDA_ROOT"]) / "nvvm" / "libdevice" / "libdevice.10.bc" if os.environ.get("MAMMODIFFUSION_CUDA_ROOT") else None,
    _Path("/usr/local/cuda/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.4/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.2/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/lib/nvidia-cuda-toolkit/nvvm/libdevice/libdevice.10.bc"),
]

_cuda_data_dir = None
for _candidate in _libdevice_candidates:
    if _candidate is not None and _candidate.exists():
        _cuda_data_dir = _candidate.parent.parent.parent
        break

if _cuda_data_dir is not None:
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={_cuda_data_dir}"
else:
    print("[WARN] libdevice.10.bc not found; if TensorFlow JIT crashes, set XLA_FLAGS manually.")

print("TF_XLA_FLAGS: ", os.environ.get("TF_XLA_FLAGS"))
print("XLA_FLAGS:    ", os.environ.get("XLA_FLAGS", "<not set>"))

# Inherit CUDA visibility by default; optionally override it for training subprocesses.
TRAIN_GPU_VISIBLE_DEVICES = os.environ.get("MAMMODIFFUSION_TRAIN_GPU")

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Multi-GPU generation does not affect training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command

TF_XLA_FLAGS:  --tf_xla_auto_jit=0
XLA_FLAGS:     --xla_gpu_cuda_data_dir=/home/fede/miniforge3/envs/tf-gpu


## 2. Dependency bootstrap

Dependency installation is disabled by default through `INSTALL_DEPENDENCIES=False`. When intentionally enabled while preparing a new environment, the cell installs the numerical, TensorFlow, PyTorch, Diffusers, and generative-evaluation packages used by G08. It writes no scientific result. A publication rerun should retain the default in an already provisioned environment and capture resolved versions whenever the dependency set is changed.


In [2]:
import subprocess, sys
# The project environment should normally be provisioned from requirements.txt.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'pillow',
        'tensorflow', 'scikit-image', 'scipy', 'psutil', 'codecarbon',
        'torch', 'diffusers', 'transformers', 'safetensors', 'accelerate',
        'prdc', 'torch-fidelity',
    ])
else:
    print('Dependencies already provisioned; installation skipped.')

Dependencies already provisioned; installation skipped.


## 3. Experiment identity, VAE option, and phase flags

The setup defines G08's canonical directories and centralizes `UNET_VERSION="v3"`, `PARAMETERIZATION="v"`, `USE_MIN_SNR=True`, and `MIN_SNR_GAMMA=5.0` so every downstream subprocess receives the same identity. `USE_VAE_FT_FROM_03` is explicit: when enabled, only a validated Diffusers VAE directory from notebook 03 is accepted; there is no silent fallback to the original codec.

Every phase is a boolean that starts `False`, so heavy training and a full image pool never begin without an explicit decision; latent preparation follows `RUN_TRAINING_PHASE`, while the preceding bootstrap cell may still create canonical directories or validate shared assets. Logs, checkpoints, latent caches, class-specific synthetic pools, metrics, figures, and energy records are separated by canonical path. These controls prevent accidental parameterization drift or overwrite, but a single G08 run still provides no estimate of training-seed variability.


In [3]:
from pathlib import Path
import shutil
import json


PROJECT_NAME = 'MammoDiffusion'
EXPERIMENT_NAME = 'diffusers/08_ldm_v3_sdvae_fromscratch'
RESULTS_STAGE_NAME = '2_diffusers/08_ldm_v3_sdvae_fromscratch'
NOTEBOOK_NAME = '08_LDM_v3_SDVAE_FromScratch.ipynb'

PROJECT_ROOT_OVERRIDE = None

# Explicit flags passed to train_ldm.py / generate_ldm.py / evaluate_ldm.py.
# Define these once and reuse them in every cell below, eliminating
# the risk that training and generation/evaluation drift out of sync.
UNET_VERSION = 'v3'
PARAMETERIZATION = 'v'
USE_MIN_SNR = True
MIN_SNR_GAMMA = 5.0

# Notebook 08 option: use the fine-tuned VAE produced by notebook 03 instead of the
# standard SD VAE. When True, path resolution checks multiple candidates and fails
# and fail explicitly when no candidate exists; there is no silent fallback.
USE_VAE_FT_FROM_03 = False
EXPERIMENT_03_NAME = 'diffusers/03_sd21_vae_finetuned'

def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        # Repository markers first: the parent directory shares the project name.
        if (candidate / 'notebooks').is_dir() and ((candidate / '.git').exists() or (candidate / 'configs').is_dir() or (candidate / 'data').is_dir()):
            return candidate
        if candidate.name == project_name:
            return candidate
    for candidate in [cwd / project_name, Path('/content') / project_name, Path.home() / project_name]:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError('MammoDiffusion root not found.')

PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
UTILITY_DIR = NOTEBOOKS_DIR / 'utility'
DATA_DIR = PROJECT_ROOT / 'data'
DATA_PROCESSED_DIR = DATA_DIR / 'processed'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_NAME
EXPERIMENT_03_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_03_NAME
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / 'pretrained_model'
PRETRAINED_MODEL_DIR = SHARED_PRETRAINED_ROOT / 'stable-diffusion-2-1-base'
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / 'archives' / 'stable-diffusion-2-1-base.zip'
MODELS_DIR = EXPERIMENT_DIR / 'models'
CHECKPOINTS_DIR = EXPERIMENT_DIR / 'checkpoints_ldm'
LATENTS_DIR = EXPERIMENT_DIR / 'latents'
LOGS_DIR = EXPERIMENT_DIR / 'logs'
RESULTS_DIR = PROJECT_ROOT / 'results' / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / 'plots'
RESULTS_METRICS_DIR = RESULTS_DIR / 'metrics'
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / 'ecotracker'
SYNTHETIC_V3_DIR = DATA_DIR / 'synthetic' / '08_ldm_v3_sdvae_fromscratch'
SYNTHETIC_V3_POS_DIR = SYNTHETIC_V3_DIR / 'positive'
SYNTHETIC_V3_NEG_DIR = SYNTHETIC_V3_DIR / 'negative'

for directory in [
    EXPERIMENT_DIR, PRETRAINED_MODEL_ZIP_PATH.parent, MODELS_DIR,
    CHECKPOINTS_DIR, LATENTS_DIR, LOGS_DIR,
    RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR, RESULTS_ECOTRACKER_DIR,
    SYNTHETIC_V3_POS_DIR, SYNTHETIC_V3_NEG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def vae_ft_03_candidate_dirs():
    """Return candidate paths, in preference order, for 03's fine-tuned VAE. Do not hard-code
    one path because 03 may save its result in different locations depending on whether it was
    resumed or launched from scratch."""
    return [
        EXPERIMENT_03_DIR / 'vae_finetuning_resume_last' / 'vae_finetuned',
        EXPERIMENT_03_DIR / 'vae_finetuning' / 'vae_finetuned',
    ]

def resolve_vae_ft_03_dir():
    """Return the first candidate containing a config.json file in Diffusers format,
    or None if none is valid. There is no silent fallback; the caller decides what to do
    when no valid candidate exists."""
    for candidate in vae_ft_03_candidate_dirs():
        if (candidate / 'config.json').is_file():
            return candidate
    return None

print('PROJECT_ROOT   :', PROJECT_ROOT)
print('EXPERIMENT_DIR :', EXPERIMENT_DIR)
print('RESULTS_DIR    :', RESULTS_DIR)
print('SYNTHETIC v3   :', SYNTHETIC_V3_DIR)
print('UNET_VERSION   :', UNET_VERSION)
print('PARAMETERIZATION:', PARAMETERIZATION)
print('USE_MIN_SNR    :', USE_MIN_SNR, '| MIN_SNR_GAMMA:', MIN_SNR_GAMMA)
print('USE_VAE_FT_FROM_03:', USE_VAE_FT_FROM_03)

if str(UTILITY_DIR) not in sys.path:
    sys.path.insert(0, str(UTILITY_DIR))
print("Inherited CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Requested GENERATION_GPU_DEVICES:", GENERATION_GPU_DEVICES)
from parallel_generation_utils import print_gpu_resolution_dry_run
print_gpu_resolution_dry_run(GENERATION_GPU_DEVICES, GENERATION_MAX_WORKERS)

PROJECT_ROOT   : /mnt/MammoDiffusion/MammoDiffusion
EXPERIMENT_DIR : /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch
RESULTS_DIR    : /mnt/MammoDiffusion/MammoDiffusion/results/2_diffusers/08_ldm_v3_sdvae_fromscratch
SYNTHETIC v3   : /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/08_ldm_v3_sdvae_fromscratch
UNET_VERSION   : v3
PARAMETERIZATION: v
USE_MIN_SNR    : True | MIN_SNR_GAMMA: 5.0
USE_VAE_FT_FROM_03: False
Inherited CUDA_VISIBLE_DEVICES: None
Requested GENERATION_GPU_DEVICES: auto
Physical GPUs (nvidia-smi): ['0', '1']
Inherited CUDA_VISIBLE_DEVICES: None
Requested GPUs (--generation-gpus): auto
Resolved GPUs: ['0', '1']
Worker count: 2


['0', '1']

In [4]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating, latents rebuilt without retraining.
#
#   RUN_LATENT_ENCODING_PHASE  encode the VAE latents that training consumes; heavy only when they are missing or stale
#   RUN_TRAINING_PHASE         train the model (hours of GPU)
#   RUN_GENERATION_PHASE       sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE       score checkpoints and record the selection
#   RUN_FILTER_PHASE           re-run the adaptive filter over the existing RAW pool
#   RUN_VALIDATION_PHASE       RAW-vs-filtered comparison; keeps its own content-aware cache
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_LATENT_ENCODING_PHASE = False
RUN_TRAINING_PHASE = False
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False  # the saved selection is consumed, not recomputed
RUN_FILTER_PHASE = False
RUN_VALIDATION_PHASE = False

## 4. Split-metadata preflight

The cell requires the canonical train, validation, and test CSV files and prints class counts for each partition. Missing metadata aborts before latent preparation. Training and model selection use training/validation data only; the test split is checked only for structural completeness and remains reserved for final classifier evaluation.

This is a structural preflight rather than a content-signature audit. The latent helper subsequently validates image paths and hashes the training/validation metadata, while patient-level isolation remains an upstream preprocessing guarantee.


In [5]:
import pandas as pd

required = [
    DATA_PROCESSED_DIR / 'metadata' / 'train.csv',
    DATA_PROCESSED_DIR / 'metadata' / 'val.csv',
    ]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Missing preprocessed dataset: {path}')

for split in ['train', 'val']:
    df = pd.read_csv(DATA_PROCESSED_DIR / 'metadata' / f'{split}.csv')
    print(split, len(df), df['label'].value_counts().to_dict())

train 2041 {0: 1701, 1: 340}
val 437 {0: 364, 1: 73}


## 5. Immutable SD 2.1 base and explicit VAE source

`verify_shared_sd21_base` confirms that the shared local SD 2.1 snapshot contains the required Diffusers components. With the default configuration, `SD_VAE_MODEL` points to that immutable snapshot and `VAE_SOURCE` is recorded as `sd_vae_original`. If the optional notebook-03 codec is requested, the code searches the documented candidate directories, requires `config.json`, and fails closed when no valid VAE exists.

The shared base is not modified or copied into G08, and its U-Net weights are not used to initialize the G08 denoiser. Recording the VAE source is necessary because changing the codec changes the latent distribution and invalidates cached latents and direct model comparisons.


In [6]:
from shared_diffusers_assets import verify_shared_sd21_base

# Keep the base SD2.1 model immutable and store it physically only once.
PRETRAINED_MODEL_DIR = verify_shared_sd21_base(PRETRAINED_MODEL_DIR)
if USE_VAE_FT_FROM_03:
    vae_ft_dir = resolve_vae_ft_03_dir()
    if vae_ft_dir is None:
        candidates_str = "\n  - ".join(str(path) for path in vae_ft_03_candidate_dirs())
        raise FileNotFoundError(
            "Fine-tuned VAE from 03 not found. Run 03_SD21_VAE_FineTuned.ipynb first."
            f"\n  - {candidates_str}"
        )
    SD_VAE_MODEL = vae_ft_dir
    VAE_SOURCE = "sd_vae_finetuned_03"
else:
    SD_VAE_MODEL = PRETRAINED_MODEL_DIR
    VAE_SOURCE = "sd_vae_original"

print("SD_VAE_MODEL:", SD_VAE_MODEL)
print("VAE_SOURCE  :", VAE_SOURCE)

SD_VAE_MODEL: /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base
VAE_SOURCE  : sd_vae_original


## 6. Manifest-bound latent encoding

`prepare_sdvae_latents.py` encodes the augmented training set and the unaugmented validation set with the explicitly selected VAE, using batches of four. It writes compressed train/validation latent arrays, per-channel latent statistics, VAE metadata, and `latents_manifest.json`; test images are excluded.

The manifest binds the cache to hashes of the train/validation CSV files and augmentation metadata, the VAE path, counts, shapes, and augmentation policy. Existing latent files are reused only when the complete manifest matches exactly, unless `FORCE_LATENTS_RECOMPUTE=True` is set. The full subprocess stream is retained in `prepare_sdvae_latents.log`.


In [7]:
def run_and_stream(cmd, log_path, env=None):
    print(' '.join(str(c) for c in cmd))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env or os.environ.copy())
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        for line in iter(process.stdout.readline, ''):
            handle.write(line)
            print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)

SDVAE_BATCH_SIZE = 4
FORCE_LATENTS_RECOMPUTE = False

prepare_cmd = [
    sys.executable,
    str(UTILITY_DIR / 'prepare_sdvae_latents.py'),
    '--project-root', str(PROJECT_ROOT),
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--batch-size', str(SDVAE_BATCH_SIZE),
    '--sd-vae-model', str(SD_VAE_MODEL),
]
if FORCE_LATENTS_RECOMPUTE:
    prepare_cmd.append('--force-recompute')

if RUN_LATENT_ENCODING_PHASE:
    run_and_stream(prepare_cmd, LOGS_DIR / 'prepare_sdvae_latents.log')
else:
    print('Reusing the existing latent cache: RUN_LATENT_ENCODING_PHASE = False.')

Reusing the existing latent cache: RUN_LATENT_ENCODING_PHASE = False.


## 7. v3 U-Net training with v-prediction and Min-SNR

When `RUN_TRAINING_PHASE` is set, `train_ldm.py` initializes the v3 conditional U-Net from scratch and optimizes for 150,000 steps, saving a Keras checkpoint every 5,000 steps. The command reuses manifest-validated latents and explicitly passes `--unet-version v3`, `--parameterization v`, `--use-min-snr`, `--min-snr-gamma 5.0`, the VAE source, and the notebook identity. `--resume-from-latest` restores the latest model, inferred step, and loss history; resumed execution should not be assumed bitwise equivalent to uninterrupted optimization.

`training_manifest.json` records the architecture version, parameterization, Min-SNR configuration, VAE source, target step, and terminal model path. That manifest prevents loading $\hat v_\theta$ as though it predicted $\hat\epsilon_\theta$. Training loss and the architectural rationale are not selection criteria; the checkpoint is chosen by the validation sweep below, and the absence of multiple independent seeds limits claims about convergence stability.


In [8]:
# IDEMPOTENT_GUARD_V1:training
if RUN_TRAINING_PHASE:
    TOTAL_STEPS = 150_000
    CHECKPOINT_EVERY = 5_000
    LOG_EVERY = 20
    RESUME_FROM_LATEST = True

    train_cmd = [
        sys.executable,
        str(UTILITY_DIR / 'train_ldm.py'),
        '--project-root', str(PROJECT_ROOT),
        '--experiment-dir', str(EXPERIMENT_DIR),
        '--total-steps', str(TOTAL_STEPS),
        '--checkpoint-every', str(CHECKPOINT_EVERY),
        '--log-every', str(LOG_EVERY),
        '--results-stage-name', RESULTS_STAGE_NAME,
        '--skip-latent-encoding',
        '--unet-version', UNET_VERSION,
        '--parameterization', PARAMETERIZATION,
        '--vae-source', VAE_SOURCE,
        '--notebook-name', NOTEBOOK_NAME,
    ]
    if USE_MIN_SNR:
        train_cmd.append('--use-min-snr')
        train_cmd.extend(['--min-snr-gamma', str(MIN_SNR_GAMMA)])
    if USE_VAE_FT_FROM_03:
        train_cmd.append('--uses-vae-ft-from-03')
    if RESUME_FROM_LATEST:
        train_cmd.append('--resume-from-latest')

    run_and_stream(train_cmd, LOGS_DIR / 'ldm_train_v3.log', env=training_subprocess_env())

## 8. Parameterization-consistent evaluation and generation

Every remaining subprocess receives the same v3 architecture, v-prediction, VAE-source, and notebook-name arguments used for training. Evaluation and generation therefore apply the required v-to-epsilon conversion rather than relying on the epsilon-prediction defaults used by earlier LDM experiments. Each command is logged and a non-zero exit stops the notebook.

Checkpoint samples, metrics, sampling-sweep records, selected pools, filter reports, final class-specific comparisons, and EcoTracker events remain in the G08 namespace. Helper-level image and checkpoint signatures provide resume safety. Consistency of implementation is necessary but does not establish that G08 is more accurate or clinically realistic than G07.


### 8.1 Validation checkpoint sweep

The `evaluate_ldm.py` sweep screens eligible G08 checkpoints using 100 generated images per class, 100 denoising steps, guidance scale 1.5, the selected SD-VAE backend, and explicit v3/v-prediction flags. It computes FID, Inception Score, and PRDC and writes the registered validation selection.

The saved validation checkpoint selection is reused by default. Set `RUN_EVALUATION_PHASE=True` with `EVAL_FORCE_RECOMPUTE=True` to run the validation sweep again and generate new manifests; sweep directories written before per-directory generation manifests existed are not rerun automatically.

The notebook validates `evaluation/best_checkpoint.json`, reroots its periodic checkpoint filename through the current project directory, and consumes that file instead of the mutable `ldm_unet_best_eval.keras` convenience copy. The positive-class FID rule remains subject to Monte Carlo noise, checkpoint multiplicity, and domain mismatch; reusing it verifies checkpoint consumption, not a new metric estimate.

In [9]:
# IDEMPOTENT_GUARD_V1:evaluation
EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_INCEPTION_BATCH = 8
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True
EVAL_FORCE_RECOMPUTE = False  # ignore cached metrics and reevaluate

if RUN_EVALUATION_PHASE:

    eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "both",
        "--min-step", str(EVAL_MIN_STEP),
        "--n-gen-per-class", str(N_GEN_PER_CLASS),
        "--sample-steps", str(EVAL_SAMPLE_STEPS),
        "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
        "--mini-batch", "1",
        "--inception-batch", str(EVAL_INCEPTION_BATCH),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
        "--sd-vae-model", str(SD_VAE_MODEL),
        "--unet-version", UNET_VERSION,
        "--parameterization", PARAMETERIZATION,
        "--vae-source", VAE_SOURCE,
        "--notebook-name", NOTEBOOK_NAME,
    ]
    if USE_VAE_FT_FROM_03:
        eval_cmd.append("--uses-vae-ft-from-03")
    if EVAL_FORCE_RECOMPUTE:
        eval_cmd.append("--force-recompute")
    if EVAL_DECODE_ON_CPU:
        eval_cmd.append("--decode-on-cpu")
    if EVAL_ECO_TRACK:
        eval_cmd.append("--eco-track")

    add_generation_parallel_args(eval_cmd)
    run_and_stream(eval_cmd, LOGS_DIR / "ldm_evaluate_sdvae_v3.log")

G08_SELECTION_PATH = EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json"
if not G08_SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Frozen G08 selection is unavailable: {G08_SELECTION_PATH}")
with G08_SELECTION_PATH.open(encoding="utf-8") as handle:
    G08_SELECTION = json.load(handle)
recorded_checkpoint = Path(G08_SELECTION["best_checkpoint"])
G08_SELECTED_CHECKPOINT = CHECKPOINTS_DIR / recorded_checkpoint.name
if not G08_SELECTED_CHECKPOINT.is_file() or G08_SELECTED_CHECKPOINT.stat().st_size == 0:
    raise FileNotFoundError(
        f"Frozen G08 periodic checkpoint is unavailable: {G08_SELECTED_CHECKPOINT}"
    )
print("Frozen G08 selection:", G08_SELECTION["best_checkpoint_id"])
print("Portable checkpoint path:", G08_SELECTED_CHECKPOINT)

Frozen G08 selection: step_75000
Portable checkpoint path: /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/checkpoints_ldm/ldm_step075000.keras


### 8.2 Exploratory sampler-step sweep on the selected checkpoint

Under the explicit `EVAL_FORCE_RECOMPUTE` policy, this cell evaluates 25, 50, 75, and 100 denoising steps on the single selected checkpoint using 50 generated images per class. With the flags off by default this exploratory recomputation is skipped. Per-budget metric JSON files are cached under `evaluation/sampling_sweep/`; the baseline checkpoint metrics are backed up before each forced recomputation and restored through `finally`, including when a sweep subprocess fails. The cell writes `sampling_sweep_summary.csv` and `recommendation.json`, choosing the smallest budget whose FID is no more than 5% above the 100-step reference.

The recommendation is a deterministic engineering heuristic, not a statistical non-inferiority test. Fifty images per class yield a noisy FID estimate, no comparable wall-time is recorded (`seconds_per_image` remains unavailable), and the same validation set is reused after checkpoint screening. The resulting choice is therefore exploratory and potentially optimistic. It does not automatically modify the generation cells, which retain 100 steps unless a reviewer explicitly accepts and documents a change.


In [10]:
# IDEMPOTENT_GUARD_V1:evaluation
if RUN_EVALUATION_PHASE:
    # Optimize the sampler on the best checkpoint.
    # evaluate_ldm.py saves metrics in the v2 checkpoints-plus-selection schema.

    RUN_SAMPLING_SWEEP = True
    SAMPLING_STEP_BUDGETS = [25, 50, 75, 100]
    SAMPLING_N_GEN_PER_CLASS = 50   # 50 images/class support the intended relative comparison
    SAMPLING_QUALITY_TOLERANCE = 1.05  # Accept a budget when FID <= 1.05 * FID at 100 steps

    evaluation_dir = EXPERIMENT_DIR / "evaluation"
    checkpoint_metrics_path = evaluation_dir / "checkpoint_metrics.json"
    sampling_sweep_dir = evaluation_dir / "sampling_sweep"
    sampling_sweep_dir.mkdir(parents=True, exist_ok=True)
    sampling_sweep_csv = sampling_sweep_dir / "sampling_sweep_summary.csv"

    GEN_SAMPLE_STEPS_RECOMMENDED = None

    if not RUN_SAMPLING_SWEEP:
        print("Sampling sweep disabled (RUN_SAMPLING_SWEEP=False); skipping.")
    elif not checkpoint_metrics_path.is_file():
        print(f"checkpoint_metrics.json not found under {checkpoint_metrics_path}. "
              "Run cell 8.1 first, then return here.")
    else:
        import pandas as pd

        with checkpoint_metrics_path.open(encoding="utf-8") as handle:
            eval_payload = json.load(handle)

        if eval_payload.get("schema_version") != 2:
            raise RuntimeError("checkpoint_metrics.json does not use the v2 schema; rerun cell 8.1.")
        selection = eval_payload.get("selection", {})
        best_checkpoint_id = selection.get("best_checkpoint_id")
        selection_metric = selection.get("selection_metric")
        checkpoints = eval_payload.get("checkpoints", [])
        if not best_checkpoint_id or not selection_metric or not checkpoints:
            raise RuntimeError("checkpoint_metrics.json v2 is incomplete; rerun cell 8.1.")
        try:
            best_row = next(row for row in checkpoints if row["checkpoint_id"] == best_checkpoint_id)
            fid_at_100 = float(best_row[selection_metric])
        except (KeyError, StopIteration, TypeError) as exc:
            raise RuntimeError("Best-checkpoint metrics are missing; rerun cell 8.1.") from exc
        best_step = int(selection.get("best_step") or best_checkpoint_id.removeprefix("step_"))
        print(f"Best checkpoint from 8.1: {best_checkpoint_id} ({selection_metric} @ {EVAL_SAMPLE_STEPS} step = {fid_at_100:.3f})")

        # The sweep command temporarily overwrites checkpoint_metrics.json.
        # Restore the copy below on completion without losing the 8.1 baseline.
        baseline_metrics_backup = sampling_sweep_dir / "baseline_checkpoint_metrics.json"
        shutil.copy2(checkpoint_metrics_path, baseline_metrics_backup)
        records = []
        for steps in SAMPLING_STEP_BUDGETS:
            target_metrics_json = sampling_sweep_dir / f"steps_{steps}_checkpoint_metrics.json"
            if target_metrics_json.is_file():
                print(f"[{steps} step] already computed -> reusing {target_metrics_json.name}")
            else:
                print(f"[{steps} step] running evaluate_ldm.py --checkpoint-id {best_checkpoint_id} --sample-steps {steps}")
                sweep_eval_cmd = [
                    sys.executable,
                    str(UTILITY_DIR / "evaluate_ldm.py"),
                    "--project-root", str(PROJECT_ROOT),
                    "--experiment-dir", str(EXPERIMENT_DIR),
                    "--mode", "both",
                    "--checkpoint-id", best_checkpoint_id,
                    "--n-gen-per-class", str(SAMPLING_N_GEN_PER_CLASS),
                    "--sample-steps", str(steps),
                    "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
                    "--mini-batch", "1",
                    "--inception-batch", str(EVAL_INCEPTION_BATCH),
                    "--results-stage-name", f"{RESULTS_STAGE_NAME}_sampling_sweep_{steps}",
                    "--vae-backend", "sd",
                    "--sd-vae-model", str(SD_VAE_MODEL),
                    "--unet-version", UNET_VERSION,
                    "--parameterization", PARAMETERIZATION,
                    "--vae-source", VAE_SOURCE,
                    "--notebook-name", NOTEBOOK_NAME,
                    "--force-recompute",
                ]
                if USE_VAE_FT_FROM_03:
                    sweep_eval_cmd.append("--uses-vae-ft-from-03")
                add_generation_parallel_args(sweep_eval_cmd)
                try:
                    run_and_stream(sweep_eval_cmd, LOGS_DIR / f"sampling_sweep_{steps}steps.log")
                    if not checkpoint_metrics_path.is_file():
                        raise RuntimeError("evaluate_ldm.py did not produce checkpoint_metrics.json.")
                    shutil.copy2(checkpoint_metrics_path, target_metrics_json)
                finally:
                    # Never leave the central selection JSON replaced by a partial sweep.
                    if baseline_metrics_backup.is_file():
                        shutil.copy2(baseline_metrics_backup, checkpoint_metrics_path)

            with target_metrics_json.open(encoding="utf-8") as handle:
                payload = json.load(handle)
            row = next(c for c in payload.get("checkpoints", []) if c.get("checkpoint_id") == best_checkpoint_id)
            wall_time = 0.0  # The v2 schema does not store comparable wall time per checkpoint.
            records.append({
                "sample_steps": steps,
                "avg_FID": float(row[selection_metric]),
                "avg_IS_mean": float(row.get("is_mean_avg", float("nan"))),
                "avg_precision": float(row.get("precision_mean", float("nan"))),
                "avg_recall": float(row.get("recall_mean", float("nan"))),
                "seconds_per_image": float(wall_time) / max(1, 2 * SAMPLING_N_GEN_PER_CLASS) if wall_time else float("nan"),
            })

        # Restore the baseline and regenerate its artifacts (CSV, plots, and manifest).
        shutil.copy2(baseline_metrics_backup, checkpoint_metrics_path)
        restore_cmd = list(eval_cmd)
        restore_cmd[restore_cmd.index("--mode") + 1] = "artifacts"
        run_and_stream(restore_cmd, LOGS_DIR / "restore_baseline_evaluation_artifacts.log")

        df_sweep = pd.DataFrame(records).sort_values("sample_steps").reset_index(drop=True)
        df_sweep.to_csv(sampling_sweep_csv, index=False)
        print("\nSampling sweep summary:")
        print(df_sweep.to_string(index=False))

        print("Sampling-sweep plots disabled; tabular metrics saved under", sampling_sweep_csv)

        # The reference must have the same image count as the sweep (50/class).
        # If 100 is not among the budgets, retain the baseline produced in 8.1.
        matching_reference = df_sweep[df_sweep["sample_steps"] == EVAL_SAMPLE_STEPS]
        if not matching_reference.empty:
            fid_at_100 = float(matching_reference["avg_FID"].iloc[0])

        # Recommend the smallest budget whose FID is within tolerance of the 100-step reference.
        tol_target = fid_at_100 * SAMPLING_QUALITY_TOLERANCE
        acceptable = df_sweep[df_sweep["avg_FID"] <= tol_target]
        if acceptable.empty:
            GEN_SAMPLE_STEPS_RECOMMENDED = int(df_sweep["sample_steps"].iloc[-1])
            print(f"\nNo budget within tolerance {SAMPLING_QUALITY_TOLERANCE:.0%} vs FID @ {EVAL_SAMPLE_STEPS} step ({fid_at_100:.3f}): "
                  f"the maximum remains recommended ({GEN_SAMPLE_STEPS_RECOMMENDED} step).")
        else:
            GEN_SAMPLE_STEPS_RECOMMENDED = int(acceptable["sample_steps"].min())
            best_row_sweep = acceptable.iloc[0]
            print(f"\nRECOMMENDATION: use {GEN_SAMPLE_STEPS_RECOMMENDED} step "
                  f"(FID {float(best_row_sweep['avg_FID']):.3f} versus reference {fid_at_100:.3f}, "
                  f"within {SAMPLING_QUALITY_TOLERANCE:.0%}). "
                  f"Change GEN_SAMPLE_STEPS in cells 8.2/8.3 if you accept the recommendation.")

        with (sampling_sweep_dir / "recommendation.json").open("w", encoding="utf-8") as handle:
            json.dump({
                "best_checkpoint_id": best_checkpoint_id,
                "best_step": int(best_step),
                "reference_sample_steps": int(EVAL_SAMPLE_STEPS),
                "reference_fid": float(fid_at_100),
                "quality_tolerance": float(SAMPLING_QUALITY_TOLERANCE),
                "step_budgets_tested": list(SAMPLING_STEP_BUDGETS),
                "recommended_gen_sample_steps": (int(GEN_SAMPLE_STEPS_RECOMMENDED)
                                                 if GEN_SAMPLE_STEPS_RECOMMENDED is not None else None),
                "sweep_summary_csv": str(sampling_sweep_csv),
            }, handle, indent=2, ensure_ascii=False)

### 8.3 Positive-class generation and filtering

Using the validation-selected G08 checkpoint, this phase completes 4,083 raw positive samples and retains 1,361 after adaptive filtering. A generation repair uses `--mode all`; a filter-only repair enters `--mode filter` directly. An independent content-aware `validate` call later verifies or reconstructs the class-scoped evidence without coupling metric completeness to image generation. Every command preserves the v3/v-prediction/VAE identity, defaults to 100 sampling steps, resumes readable indexed images, regenerates corrupt or missing files, and records content-aware manifests and sustainability events.

The final pool is written to `data/synthetic/08_ldm_v3_sdvae_fromscratch/positive/`. Filtered quality may improve while coverage decreases, and G08 was not the selected downstream generator; these outputs are comparative research artifacts, not substitutes for the selected G07 positive pool.


In [11]:
# IDEMPOTENT_GUARD_V1:generation
GEN_N_RAW = 4083
GEN_N_SELECTED = 1361
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = G08_SELECTED_CHECKPOINT
GEN_DECODE_ON_CPU = False
GEN_ECO_TRACK = True
if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    POS_EXECUTION_MODE = "all" if RUN_GENERATION_PHASE else "filter"

    pos_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", POS_EXECUTION_MODE,
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "1",
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
        "--sd-vae-model", str(SD_VAE_MODEL),
        "--unet-version", UNET_VERSION,
        "--parameterization", PARAMETERIZATION,
        "--vae-source", VAE_SOURCE,
        "--notebook-name", NOTEBOOK_NAME,
    ]
    if USE_VAE_FT_FROM_03:
        pos_cmd.append("--uses-vae-ft-from-03")
    if GEN_DECODE_ON_CPU:
        pos_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        pos_cmd.append("--eco-track")

    # --mode all writes to canonical experiment paths (get_experiment_paths), which for
    # EXPERIMENT_NAME=diffusers/08_ldm_v3_sdvae_fromscratch resolves to SYNTHETIC_V3_POS_DIR
    # (see ldm_project_paths.FILTERED_DIR_NAME_BY_EXPERIMENT) -- no collision with
    # 07.
    add_generation_parallel_args(pos_cmd)
    run_and_stream(pos_cmd, LOGS_DIR / "ldm_generate_positive_sdvae_v3.log")

### 8.4 Negative-class generation and filtering

The same checkpoint, codec, v-prediction conversion, guidance scale, sampling budget, raw target, and retained target are applied to label 0 in separate raw and filtered directories. Generation repair uses `--mode all`, filter-only repair uses `--mode filter`, and independent cache checks verify the validation metrics. The resulting 1,361-image negative pool is stored at `data/synthetic/08_ldm_v3_sdvae_fromscratch/negative/`, with class-scoped metrics and logs.

Symmetric execution supports a complete two-class audit, but class-specific distribution metrics do not establish diagnostic discrimination. Results remain descriptive and cannot be used to tune G08 or overturn the protocol-defined generator-selection process.


In [12]:
# IDEMPOTENT_GUARD_V1:generation
if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    NEG_EXECUTION_MODE = "all" if RUN_GENERATION_PHASE else "filter"
    NEG_RAW_DIR = EXPERIMENT_DIR / "synthetic_raw_negative"
    NEG_FILTERED_DIR = SYNTHETIC_V3_NEG_DIR

    neg_base_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "0",
        "--raw-dir", str(NEG_RAW_DIR),
        "--filtered-dir", str(NEG_FILTERED_DIR),
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
        "--sd-vae-model", str(SD_VAE_MODEL),
        "--unet-version", UNET_VERSION,
        "--parameterization", PARAMETERIZATION,
        "--vae-source", VAE_SOURCE,
        "--notebook-name", NOTEBOOK_NAME,
    ]
    if USE_VAE_FT_FROM_03:
        neg_base_cmd.append("--uses-vae-ft-from-03")
    if GEN_DECODE_ON_CPU:
        neg_base_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        neg_base_cmd.append("--eco-track")

    add_generation_parallel_args(neg_base_cmd)
    neg_cmd = [*neg_base_cmd, "--mode", NEG_EXECUTION_MODE]
    run_and_stream(neg_cmd, LOGS_DIR / "ldm_negative_sdvae_v3_all.log")

### 8.5 Artifact summary

The final cell invokes content-aware validation cache checks for both classes without entering generation, then counts the PNG files in each filtered pool. A complete run should expose 1,361 retained images for each label.

This is a presence audit, not a checksum or statistical validation. Reuse or publication additionally requires the matching latent/runtime/generation/filter manifests, content signatures, logs, and unified benchmark records. The summary must preserve G08's status as an experimental alternative rather than the selected from-scratch generator.

In [13]:
from pathlib import Path
import pandas as pd

def verify_g08_validation_cache(target_label, raw_dir, filtered_dir, log_prefix):
    command = [
        sys.executable, str(UTILITY_DIR / 'generate_ldm.py'), '--project-root', str(PROJECT_ROOT),
        '--experiment-dir', str(EXPERIMENT_DIR), '--model-path', str(GEN_MODEL_PATH),
        '--n-raw', str(GEN_N_RAW), '--n-selected', str(GEN_N_SELECTED),
        '--target-label', str(target_label), '--raw-dir', str(raw_dir), '--filtered-dir', str(filtered_dir),
        '--sample-steps', str(GEN_SAMPLE_STEPS), '--guidance-scale', str(GEN_GUIDANCE_SCALE),
        '--results-stage-name', RESULTS_STAGE_NAME, '--vae-backend', 'sd', '--sd-vae-model', str(SD_VAE_MODEL),
        '--unet-version', UNET_VERSION, '--parameterization', PARAMETERIZATION,
        '--vae-source', VAE_SOURCE, '--notebook-name', NOTEBOOK_NAME, '--mode', 'validate',
    ]
    if USE_VAE_FT_FROM_03: command.append('--uses-vae-ft-from-03')
    if GEN_DECODE_ON_CPU: command.append('--decode-on-cpu')
    if GEN_ECO_TRACK: command.append('--eco-track')
    add_generation_parallel_args(command)
    run_and_stream(command, LOGS_DIR / f'{log_prefix}_validation.log')

if RUN_VALIDATION_PHASE:
    verify_g08_validation_cache(1, EXPERIMENT_DIR / 'synthetic_raw_positive', SYNTHETIC_V3_POS_DIR, 'ldm_positive_sdvae_v3')
    verify_g08_validation_cache(0, EXPERIMENT_DIR / 'synthetic_raw_negative', SYNTHETIC_V3_NEG_DIR, 'ldm_negative_sdvae_v3')

pd.DataFrame([
    {'class': name, 'directory': str(directory), 'n_png': len(list(Path(directory).glob('*.png'))) if Path(directory).is_dir() else 0}
    for name, directory in [('positive', SYNTHETIC_V3_POS_DIR), ('negative', SYNTHETIC_V3_NEG_DIR)]
])

,class,directory,n_png
0,positive,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
1,negative,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
